In [ ]:
"""
FedAvg on Wheat Disease Dataset
================================
Federated Averaging (McMahan et al., 2017) implementation for the Wheat
leaf disease classification task, matching the fixed experimental
configuration used across all FL algorithm comparisons in this project.

Config:
    Clients            : 5
    Partitioning        : Dirichlet non-IID, alpha = 0.5
    Backbone            : ResNet18 (ImageNet-pretrained, fc replaced)
    Optimizer           : SGD, lr=0.001, momentum=0.9
    Local epochs/round  : 5
    Communication rounds: 10
    Batch size          : 32
    Dataset             : Wheat -> only the `train/` subfolder is used

Kaggle-safety features:
    - Per-round metrics appended to a CSV (survives partial runs)
    - Rolling JSON checkpoint with global model weights + round number
    - Confusion matrix dumped as a formatted .txt after every round
    - All outputs versioned under /kaggle/working/
"""

import os
import copy
import json
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix
)

# ----------------------------------------------------------------------
# 0. CONFIG
# ----------------------------------------------------------------------
SEED = 42
NUM_CLIENTS = 5
DIRICHLET_ALPHA = 0.5
NUM_ROUNDS = 10
LOCAL_EPOCHS = 5
BATCH_SIZE = 32
LR = 0.001
MOMENTUM = 0.9
IMG_SIZE = 224
VAL_SPLIT = 0.2  # held-out from the same train/ folder for evaluation

# NOTE: update this to your actual Kaggle dataset path.
# Only the `train/` subfolder of the Wheat dataset should be used.
DATASET_ROOT = "/kaggle/input/datasets/kushagra3204/wheat-plant-diseases/data/train"

OUTPUT_DIR = "/kaggle/working/fedavg_wheat"
os.makedirs(OUTPUT_DIR, exist_ok=True)
CSV_PATH = os.path.join(OUTPUT_DIR, "fedavg_wheat_results.csv")
CKPT_PATH = os.path.join(OUTPUT_DIR, "fedavg_wheat_checkpoint.json")
CM_DIR = os.path.join(OUTPUT_DIR, "confusion_matrices")
os.makedirs(CM_DIR, exist_ok=True)

# Strict CPU-storage / GPU-compute contract:
# All persisted state (client states, global state, checkpoints) lives on
# CPU. Tensors are moved to GPU only inside the train/eval loops.
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


# ----------------------------------------------------------------------
# 1. DATA LOADING
# ----------------------------------------------------------------------
def build_transforms():
    train_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                              std=[0.229, 0.224, 0.225]),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                              std=[0.229, 0.224, 0.225]),
    ])
    return train_tf, eval_tf


class TransformSubset(Dataset):
    """Wraps a Subset so train/eval subsets can use different transforms."""
    def __init__(self, base_dataset, indices, transform):
        self.base_dataset = base_dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        path, label = self.base_dataset.samples[real_idx]
        img = self.base_dataset.loader(path)
        if self.transform:
            img = self.transform(img)
        return img, label


def load_wheat_dataset():
    train_tf, eval_tf = build_transforms()
    full_dataset = datasets.ImageFolder(DATASET_ROOT)
    class_names = full_dataset.classes
    num_classes = len(class_names)

    targets = np.array([s[1] for s in full_dataset.samples])
    all_indices = np.arange(len(full_dataset))

    rng = np.random.default_rng(SEED)
    rng.shuffle(all_indices)

    val_size = int(len(all_indices) * VAL_SPLIT)
    val_indices = all_indices[:val_size]
    train_indices = all_indices[val_size:]

    train_wrapped = TransformSubset(full_dataset, train_indices, train_tf)
    val_wrapped = TransformSubset(full_dataset, val_indices, eval_tf)

    return full_dataset, train_wrapped, val_wrapped, train_indices, targets, num_classes, class_names


def dirichlet_partition(train_indices, targets, num_clients, alpha, seed=SEED):
    """
    Partition `train_indices` across `num_clients` using a Dirichlet(alpha)
    distribution per class, producing non-IID client splits.
    Returns: dict {client_id: [local positions into the train subset]}
    """
    rng = np.random.default_rng(seed)
    train_targets = targets[train_indices]
    classes = np.unique(train_targets)

    client_indices = {i: [] for i in range(num_clients)}

    for c in classes:
        class_positions = np.where(train_targets == c)[0]
        rng.shuffle(class_positions)

        proportions = rng.dirichlet(alpha=[alpha] * num_clients)
        proportions = (np.cumsum(proportions) * len(class_positions)).astype(int)[:-1]

        split_positions = np.split(class_positions, proportions)
        for client_id, positions in enumerate(split_positions):
            client_indices[client_id].extend(positions.tolist())

    for client_id in client_indices:
        rng.shuffle(client_indices[client_id])

    return client_indices


# ----------------------------------------------------------------------
# 2. MODEL
# ----------------------------------------------------------------------
def build_model(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


# ----------------------------------------------------------------------
# 3. LOCAL TRAINING (client update)
# ----------------------------------------------------------------------
def local_train(global_state_cpu, client_loader, num_classes, local_epochs=LOCAL_EPOCHS):
    model = build_model(num_classes)
    model.load_state_dict(global_state_cpu)
    model.to(DEVICE)
    model.train()

    optimizer = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM)
    criterion = nn.CrossEntropyLoss()

    for _ in range(local_epochs):
        for images, labels in client_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    # Return state dict moved back to CPU (storage contract)
    cpu_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    del model
    torch.cuda.empty_cache()
    return cpu_state


# ----------------------------------------------------------------------
# 4. FEDAVG AGGREGATION
# ----------------------------------------------------------------------
def fedavg_aggregate(client_states, client_sizes):
    total_size = sum(client_sizes)
    avg_state = copy.deepcopy(client_states[0])

    for key in avg_state:
        if avg_state[key].dtype in (torch.float32, torch.float64, torch.float16):
            avg_state[key] = torch.zeros_like(avg_state[key], dtype=torch.float32)
            for state, size in zip(client_states, client_sizes):
                weight = size / total_size
                avg_state[key] += state[key].float() * weight
            avg_state[key] = avg_state[key].to(client_states[0][key].dtype)
        else:
            # Non-float buffers (e.g. num_batches_tracked): weighted-majority
            # is meaningless, so just take the largest client's value.
            largest_idx = int(np.argmax(client_sizes))
            avg_state[key] = client_states[largest_idx][key].clone()

    return avg_state


# ----------------------------------------------------------------------
# 5. EVALUATION
# ----------------------------------------------------------------------
def evaluate_global_model(global_state_cpu, val_loader, num_classes, class_names, round_num):
    model = build_model(num_classes)
    model.load_state_dict(global_state_cpu)
    model.to(DEVICE)
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.numpy().tolist())

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="macro", zero_division=0
    )
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(num_classes)))

    save_confusion_matrix(cm, class_names, round_num)

    del model
    torch.cuda.empty_cache()
    return acc, precision, recall, f1


def save_confusion_matrix(cm, class_names, round_num):
    """Format depends on class count: small/medium/large, per project convention."""
    n = len(class_names)
    path = os.path.join(CM_DIR, f"round_{round_num:03d}_confusion_matrix.txt")

    with open(path, "w") as f:
        f.write(f"Confusion Matrix - Round {round_num}\n")
        f.write("=" * 60 + "\n\n")

        if n <= 6:
            # Small: full labeled grid
            header = "true\\pred".ljust(15) + "".join(c[:10].rjust(12) for c in class_names)
            f.write(header + "\n")
            for i, row in enumerate(cm):
                line = class_names[i][:14].ljust(15) + "".join(str(v).rjust(12) for v in row)
                f.write(line + "\n")
        elif n <= 15:
            # Medium: numeric indices + a legend
            f.write("Legend:\n")
            for i, c in enumerate(class_names):
                f.write(f"  [{i}] {c}\n")
            f.write("\n")
            header = "".ljust(6) + "".join(f"[{i}]".rjust(7) for i in range(n))
            f.write(header + "\n")
            for i, row in enumerate(cm):
                line = f"[{i}]".ljust(6) + "".join(str(v).rjust(7) for v in row)
                f.write(line + "\n")
        else:
            # Large: sparse listing of non-zero entries only
            f.write("Legend:\n")
            for i, c in enumerate(class_names):
                f.write(f"  [{i}] {c}\n")
            f.write("\nNon-zero entries (true_idx -> pred_idx : count):\n")
            for i in range(n):
                for j in range(n):
                    if cm[i, j] > 0:
                        f.write(f"  {i} -> {j} : {cm[i, j]}\n")


# ----------------------------------------------------------------------
# 6. RESULT PERSISTENCE (CSV append + rolling JSON checkpoint)
# ----------------------------------------------------------------------
def append_csv_result(round_num, acc, precision, recall, f1):
    row = {
        "round": round_num,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }
    df_row = pd.DataFrame([row])
    header_needed = not os.path.exists(CSV_PATH)
    df_row.to_csv(CSV_PATH, mode="a", header=header_needed, index=False)


def save_checkpoint(global_state_cpu, round_num):
    """
    Rolling JSON checkpoint. Model weights are base64-free here: we save
    a torch .pt alongside a small JSON pointer/metadata file so a Kaggle
    timeout mid-round never leaves things in an inconsistent state.
    """
    weights_path = os.path.join(OUTPUT_DIR, "fedavg_wheat_global_weights.pt")
    torch.save(global_state_cpu, weights_path)

    meta = {
        "algorithm": "FedAvg",
        "dataset": "Wheat",
        "last_completed_round": round_num,
        "total_rounds": NUM_ROUNDS,
        "weights_file": weights_path,
    }
    with open(CKPT_PATH, "w") as f:
        json.dump(meta, f, indent=2)


def try_resume():
    """Resume from checkpoint if present (Kaggle session-timeout recovery)."""
    if os.path.exists(CKPT_PATH):
        with open(CKPT_PATH, "r") as f:
            meta = json.load(f)
        weights_path = meta["weights_file"]
        if os.path.exists(weights_path):
            state = torch.load(weights_path, map_location="cpu")
            start_round = meta["last_completed_round"] + 1
            print(f"[Resume] Found checkpoint at round {meta['last_completed_round']}. "
                  f"Resuming from round {start_round}.")
            return state, start_round
    return None, 1


# ----------------------------------------------------------------------
# 7. MAIN FEDERATED LOOP
# ----------------------------------------------------------------------
def main():
    print(f"Device: {DEVICE}")
    print("Loading Wheat dataset (train/ subfolder only)...")

    (full_dataset, train_wrapped, val_wrapped,
     train_indices, targets, num_classes, class_names) = load_wheat_dataset()

    print(f"Classes ({num_classes}): {class_names}")
    print(f"Train samples: {len(train_wrapped)} | Val samples: {len(val_wrapped)}")

    val_loader = DataLoader(val_wrapped, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    # Dirichlet non-IID split of the train subset across clients.
    # `dirichlet_partition` returns positions local to `train_indices`,
    # which line up 1:1 with positions in `train_wrapped`.
    client_local_positions = dirichlet_partition(
        train_indices, targets, NUM_CLIENTS, DIRICHLET_ALPHA
    )

    client_loaders = []
    client_sizes = []
    for cid in range(NUM_CLIENTS):
        positions = client_local_positions[cid]
        subset = Subset(train_wrapped, positions)
        loader = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
        client_loaders.append(loader)
        client_sizes.append(len(positions))
        print(f"  Client {cid}: {len(positions)} samples")

    # Init or resume global model
    resumed_state, start_round = try_resume()
    if resumed_state is not None:
        global_state = resumed_state
    else:
        init_model = build_model(num_classes)
        global_state = {k: v.detach().cpu().clone() for k, v in init_model.state_dict().items()}
        del init_model

    print(f"\nStarting FedAvg from round {start_round} to {NUM_ROUNDS}...\n")

    for round_num in range(start_round, NUM_ROUNDS + 1):
        client_states = []

        for cid in range(NUM_CLIENTS):
            updated_state = local_train(
                global_state, client_loaders[cid], num_classes, LOCAL_EPOCHS
            )
            client_states.append(updated_state)

        global_state = fedavg_aggregate(client_states, client_sizes)

        acc, precision, recall, f1 = evaluate_global_model(
            global_state, val_loader, num_classes, class_names, round_num
        )

        append_csv_result(round_num, acc, precision, recall, f1)
        save_checkpoint(global_state, round_num)

        print(f"[Round {round_num:03d}/{NUM_ROUNDS}] "
              f"Acc: {acc:.4f} | Prec: {precision:.4f} | "
              f"Rec: {recall:.4f} | F1: {f1:.4f}")

    print("\nFedAvg training complete for Wheat dataset.")
    print(f"Results CSV : {CSV_PATH}")
    print(f"Checkpoint  : {CKPT_PATH}")
    print(f"Confusion matrices: {CM_DIR}")


if __name__ == "__main__":
    main()